# 🏆 Amazon ML Challenge 2026 — Business Entity Resolution

## Solution Notebook

**Task:** Given business records from 3 independent sources with noisy/incomplete data across multiple languages (US, India, France), determine which Source 2 & 3 records refer to the same real-world entity as each Source 1 record.

**Metric:** F_0.5 (macro-averaged, precision-heavy — penalises false merges 2× more than misses)

**Constraints:** MIT/Apache 2.0 licensed models, ≤8B parameters

---

## 🗺️ Pipeline Architecture

```
Raw Data ──► Text Normalisation ──► Blocking/Candidate Gen ──► Feature Scoring ──► Classification ──► Output
              (unicode, abbrevs,       (TF-IDF / FAISS)         (similarity        (LightGBM /
               links, languages)                                 features)          cross-encoder)
```

### Phase 1 (Fast Baseline — submit in hours)
- **Blocking:** TF-IDF cosine similarity on combined name+address text  
- **Classifier:** LightGBM with string similarity features (Jaccard, Levenshtein, token overlap, cosine)
- **Train time:** ~30 min on full dataset

### Phase 2 (Optimal — 1-2 days)
- **Blocking:** Multilingual Sentence Transformers + FAISS ANN search  
- **Reranker:** Cross-encoder scoring on top candidates  
- **Ensemble:** Phase 1 LightGBM + Phase 2 cross-encoder scores  
- **Train time:** ~4-12 hours depending on GPU

---

## 🔧 How to Use
1. Set `USE_DUMMY_DATA = False` and `PHASE = 1` or `2` in the Config cell
2. Run all cells — outputs go to `output/`
3. Run the validator cell at the bottom
4. Upload `output/matching_results.tsv` to the leaderboard


## 📦 0. Install Dependencies

In [ ]:
# Run this cell once to install all requirements
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'], check=True)
print('[OK] Dependencies installed')

## ⚙️ 1. Configuration

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')

# ===============================================================
#  USER-TUNABLE SETTINGS -- adjust these before each run
# ===============================================================

USE_DUMMY_DATA = True       # True -> generate a small synthetic dataset for testing
                            # False -> read from DATA_DIR (real competition files)

PHASE = 1                   # 1 -> Phase 1 only (fast baseline)
                            # 2 -> Phase 1 + Phase 2 (full pipeline, better accuracy)

# --- Data Paths -----------------------------------------------
DATA_DIR   = 'dataset'           # root; expects train/ and test/ subdirs
TRAIN_DIR  = os.path.join(DATA_DIR, 'train')
TEST_DIR   = os.path.join(DATA_DIR, 'test')
OUTPUT_DIR = 'output'
MODEL_DIR  = 'models'

for d in [OUTPUT_DIR, MODEL_DIR, TRAIN_DIR, TEST_DIR]:
    os.makedirs(d, exist_ok=True)

# --- Phase 1 Hyperparameters ----------------------------------
P1_TFIDF_MAX_FEATURES  = 100_000   # TF-IDF vocabulary size
P1_TFIDF_NGRAM_RANGE   = (1, 2)    # unigrams + bigrams
P1_TOP_K_CANDIDATES    = 50        # candidates per S1 entity from TF-IDF blocking
P1_MATCH_THRESHOLD     = 0.45      # LightGBM probability threshold (higher = more precise)
P1_LGBM_PARAMS = {
    'objective':       'binary',
    'metric':          'binary_logloss',
    'num_leaves':      127,
    'learning_rate':   0.05,
    'n_estimators':    800,
    'min_child_samples': 5,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq':    5,
    'reg_alpha':       0.1,
    'reg_lambda':      0.1,
    'random_state':    42,
    'verbose':         -1,
    'n_jobs':          -1,
}
VAL_FRACTION = 0.15    # fraction of train S1 entities held out for local evaluation

# --- Phase 2 Hyperparameters ----------------------------------
P2_EMBED_MODEL  = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'
                  # 278M params, Apache 2.0 -- handles EN/HI/FR and 48 other langs
P2_CROSS_MODEL  = 'cross-encoder/mmarco-mMiniLMv2-L12-H384-v1'
                  # 33M params, Apache 2.0 -- multilingual cross-encoder
P2_TOP_K_FAISS  = 100    # ANN candidates from FAISS per S1
P2_BATCH_SIZE   = 128    # encoding batch size
P2_MATCH_THRESHOLD = 0.40  # ensemble threshold for Phase 2

# --- Misc -----------------------------------------------------
RANDOM_SEED    = 42
MAX_TEXT_LEN   = 512     # characters to keep per text field
DEVICE         = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'

print(f'Phase: {PHASE}  |  Dummy data: {USE_DUMMY_DATA}  |  Device: {DEVICE}')

## 📚 2. Imports

In [ ]:
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')
except Exception:
    pass
import re, unicodedata, json, pickle, time
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Optional, Set, Tuple

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from scipy.sparse import csr_matrix
import lightgbm as lgb

from rapidfuzz import fuzz, distance as rf_distance
from tqdm.auto import tqdm
import joblib

try:
    import ftfy
    HAS_FTFY = True
except ImportError:
    HAS_FTFY = False

print('[OK] Core imports OK')
print(f'   LightGBM {lgb.__version__}')

## 🧹 3. Text Preprocessing Utilities

Handles: unicode normalization, mojibake, URLs/links, multi-language abbreviations, missing fields, unknown characters.

In [ ]:
# -------------------------------------------------------------
#  3a. Abbreviation expansion dictionaries (multi-language)
# -------------------------------------------------------------

NAME_ABBREVS = {
    # Legal suffixes -- English
    r'\bcorp\.?\b':          'corporation',
    r'\bco\.?\b':            'company',
    r'\binc\.?\b':           'incorporated',
    r'\bltd\.?\b':           'limited',
    r'\bllc\.?\b':           'limited liability company',
    r'\bllp\.?\b':           'limited liability partnership',
    r'\blp\.?\b':            'limited partnership',
    r'\bplc\.?\b':           'public limited company',
    r'\bint\'?l\.?\b':       'international',
    r'\bintl\.?\b':          'international',
    r'\bsvc\.?s?\b':         'services',
    r'\bmfg\.?\b':           'manufacturing',
    r'\bmgmt\.?\b':          'management',
    r'\bassoc\.?s?\b':       'associates',
    r'\bconsult\.?\b':       'consulting',
    r'\btech\.?\b':          'technology',
    r'\btech\.?nologies\b':  'technologies',
    r'\bdist\.?\b':          'distributors',
    r'\bentpr\.?\b':         'enterprises',
    r'\benterp\.?\b':        'enterprises',
    r'\bgrp\.?\b':           'group',
    r'\binds\.?\b':          'industries',
    r'\bind\.?\b':           'industries',
    r'&':                    'and',
    # Indian company suffixes
    r'\bpvt\.?\b':           'private',
    r'\bpriv\.?\b':          'private',
    r'\bundkg\.?\b':         'undertaking',
    # French suffixes
    r'\bsarl\.?\b':          'société à responsabilité limitée',
    r'\bsa\.?\b':            'société anonyme',
    r'\bsas\.?\b':           'société par actions simplifiée',
    r'\bsnc\.?\b':           'société en nom collectif',
    r'\bei\.?r?\.?l?\.?\b':  'entreprise individuelle',
}

ADDR_ABBREVS = {
    # US address abbreviations
    r'\bst\.?\b':     'street',
    r'\brd\.?\b':     'road',
    r'\bave\.?\b':    'avenue',
    r'\bblvd\.?\b':   'boulevard',
    r'\bdr\.?\b':     'drive',
    r'\bln\.?\b':     'lane',
    r'\bct\.?\b':     'court',
    r'\bpl\.?\b':     'place',
    r'\bpkwy\.?\b':   'parkway',
    r'\bhwy\.?\b':    'highway',
    r'\bfwy\.?\b':    'freeway',
    r'\bsq\.?\b':     'square',
    r'\bflr\.?\b':    'floor',
    r'\bfl\.?\b':     'floor',
    r'\bste\.?\b':    'suite',
    r'\bapt\.?\b':    'apartment',
    r'\bpob\.?\b':    'post office box',
    r'\bpo box\.?\b': 'post office box',
    r'\bn\.?\b':      'north',
    r'\bs\.?\b':      'south',
    r'\be\.?\b':      'east',
    r'\bw\.?\b':      'west',
    r'\bnw\.?\b':     'northwest',
    r'\bne\.?\b':     'northeast',
    r'\bsw\.?\b':     'southwest',
    r'\bse\.?\b':     'southeast',
    # Indian address abbreviations
    r'\bnear\b':      'near',     # keep but normalize spacing
    r'\bopp\.?\b':    'opposite',
    r'\bbeh\.?\b':    'behind',
    r'\bdist\.?\b':   'district',
    r'\btal\.?\b':    'taluka',
    r'\bvill\.?\b':   'village',
    r'\bnagar\.?\b':  'nagar',
    r'\bpin\.?\b':    '',   # drop PIN label
    # French
    r'\brue\.?\b':    'rue',
    r'\bav\.?\b':     'avenue',
    r'\bbd\.?\b':     'boulevard',
    r'\bpl\.?\b':     'place',
    r'\bappt\.?\b':   'appartement',
    r'\bcx\.?\b':     'cedex',
}

# Compile patterns once
_NAME_PATTERNS = [(re.compile(k, re.IGNORECASE), v) for k, v in NAME_ABBREVS.items()]
_ADDR_PATTERNS = [(re.compile(k, re.IGNORECASE), v) for k, v in ADDR_ABBREVS.items()]

# URL / link patterns
_URL_RE  = re.compile(r'https?://\S+|www\.\S+', re.IGNORECASE)
_EMAIL_RE = re.compile(r'\S+@\S+\.\S+')
# Unicode junk: surrogates, private-use area, replacement char
_JUNK_UNICODE_RE = re.compile(r'[\uFFFD\uFFFE\uFFFF\uD800-\uDFFF\uE000-\uF8FF]')

print('[OK] Abbreviation dictionaries compiled')

In [ ]:
# -------------------------------------------------------------
#  3b. Core preprocessing functions
# -------------------------------------------------------------

def fix_unicode(text: str) -> str:
    """Fix mojibake and normalize unicode."""
    if HAS_FTFY:
        text = ftfy.fix_text(text)
    # NFC normalisation -- canonical composition
    text = unicodedata.normalize('NFC', text)
    # Remove known junk unicode chars
    text = _JUNK_UNICODE_RE.sub(' ', text)
    return text


def remove_links(text: str) -> str:
    text = _URL_RE.sub(' ', text)
    text = _EMAIL_RE.sub(' ', text)
    return text


def normalize_punctuation(text: str) -> str:
    # Unify quote/apostrophe variants
    text = re.sub(r'[\u2018\u2019\u201A\u201B\u0060\u00B4]', "'", text)
    text = re.sub(r'[\u201C\u201D\u201E\u201F]', '"', text)
    # Unify dashes
    text = re.sub(r'[\u2012\u2013\u2014\u2015\u2212]', '-', text)
    return text


def expand_abbreviations(text: str, patterns: list) -> str:
    for pat, repl in patterns:
        text = pat.sub(repl, text)
    return text


def clean_whitespace(text: str) -> str:
    return re.sub(r'\s+', ' ', text).strip()


def preprocess_name(raw: str) -> str:
    """Full preprocessing pipeline for business names."""
    if not isinstance(raw, str) or not raw.strip():
        return ''
    text = fix_unicode(raw)
    text = remove_links(text)
    text = normalize_punctuation(text)
    text = text.lower()
    text = expand_abbreviations(text, _NAME_PATTERNS)
    # Remove non-alphanumeric except spaces (preserve unicode letters for multi-lang)
    text = re.sub(r'[^\w\s]', ' ', text, flags=re.UNICODE)
    text = clean_whitespace(text)
    return text[:MAX_TEXT_LEN]


def preprocess_address(raw: str) -> str:
    """Full preprocessing pipeline for addresses."""
    if not isinstance(raw, str) or not raw.strip():
        return ''
    text = fix_unicode(raw)
    text = remove_links(text)
    text = normalize_punctuation(text)
    text = text.lower()
    text = expand_abbreviations(text, _ADDR_PATTERNS)
    # Remove standalone numbers that are postal codes (5+ digits, or Indian 6-digit PIN)
    # Keep them as they can help matching
    text = re.sub(r'[^\w\s]', ' ', text, flags=re.UNICODE)
    text = clean_whitespace(text)
    return text[:MAX_TEXT_LEN]


def make_combined_text(name: str, address: str) -> str:
    """Combine name and address into a single search string."""
    return (name + ' ' + address).strip()


def preprocess_df(df: pd.DataFrame) -> pd.DataFrame:
    """Apply preprocessing to a source dataframe in-place, return it."""
    df = df.copy()
    df['business_name']    = df['business_name'].fillna('').astype(str)
    df['business_address'] = df['business_address'].fillna('').astype(str)
    df['country']          = df['country'].fillna('').astype(str).str.upper().str.strip()
    df['clean_name']    = df['business_name'].apply(preprocess_name)
    df['clean_address'] = df['business_address'].apply(preprocess_address)
    df['combined']      = df.apply(
        lambda r: make_combined_text(r['clean_name'], r['clean_address']), axis=1
    )
    return df


# Quick sanity check
assert preprocess_name('Acme Corp. & Sons Ltd.') == 'acme corporation and sons limited'
assert preprocess_address('123 Main St., Ste 4B') == '123 main street  suite 4b'
print('[OK] Preprocessing functions OK')

## 🗄️ 4. Synthetic Dummy Dataset (for testing)

Creates a small self-contained dataset that mirrors the real competition format. Set `USE_DUMMY_DATA = False` to skip this and use real files.

In [ ]:
def create_dummy_dataset():
    """Create small synthetic TSV files under dataset/train and dataset/test."""
    import random
    random.seed(42)

    # -- Ground-truth entities (the 'true' businesses) ---------
    TRUE_ENTITIES = [
        # (name, address, country)
        ('Acme Corporation',              '123 Main Street, Springfield, IL 62701', 'US'),
        ('Global Tech Solutions Inc',     '500 Silicon Ave, San Jose, CA 95101',    'US'),
        ('Springfield Bakery & Cafe',     '45 Baker Road, Springfield, IL 62702',   'US'),
        ('Sunrise Hotels Limited',        'Near Airport, Andheri East, Mumbai 400069', 'India'),
        ('Sharma Enterprises Pvt Ltd',    'Plot 12, Okhla Industrial Area, New Delhi 110020', 'India'),
        ('Café de Paris SARL',            '10 Rue de Rivoli, 75001 Paris, France',  'France'),
        ('TechVision International Corp', '200 Innovation Blvd, Austin, TX 78701',  'US'),
        ('Mumbai Spice Traders Private',  'Shop 5, Crawford Market, Mumbai 400001', 'India'),
        ('Lyon Textiles SAS',             '42 Avenue Jean Jaurès, 69007 Lyon, France', 'France'),
        ('DataStream Analytics LLC',      '300 Market Street Suite 1200, San Francisco, CA 94105', 'US'),
        # Singletons (no matches in S2/S3)
        ('Unique Solutions Corp',         '999 Nowhere Lane, Remote, WY 82001', 'US'),
        ('Paris Boutique SNC',            '7 Rue du Faubourg, 75008 Paris, France', 'France'),
    ]

    def perturb_name(name):
        """Apply realistic noise to a business name."""
        ops = [
            lambda s: s.replace('Corporation', 'Corp.').replace('Limited', 'Ltd').replace('Private', 'Pvt'),
            lambda s: s.replace('International', "Int'l").replace('Solutions', 'Solns'),
            lambda s: s.replace(' and ', ' & ').replace('Incorporated', 'Inc'),
            lambda s: re.sub(r'\s+', ' ', s.replace('Technologies', 'Tech').replace('Management', 'Mgmt')),
            lambda s: s.upper(),
            lambda s: s.lower(),
            lambda s: s + ' (Branch)',
            lambda s: s.replace('Enterprises', 'Enterp').replace('Associates', 'Assoc.'),
            lambda s: ' '.join(reversed(s.split())),  # word order swap
            lambda s: s,  # no change
        ]
        return random.choice(ops)(name)

    def perturb_address(addr):
        """Apply realistic noise to an address."""
        ops = [
            lambda s: re.sub(r'\bStreet\b', 'St.', s).replace('Avenue', 'Ave').replace('Boulevard', 'Blvd'),
            lambda s: re.sub(r',.*$', '', s),           # truncate after first comma
            lambda s: re.sub(r'\d{5,6}', '', s).strip(), # remove PIN/ZIP
            lambda s: s + ' (Near Main Market)',         # landmark reference
            lambda s: s.replace('Suite', 'Ste').replace('Road', 'Rd').replace('Drive', 'Dr'),
            lambda s: s.upper(),
            lambda s: '',  # missing address
            lambda s: s,
        ]
        return random.choice(ops)(addr)

    def make_source1(entities, id_prefix, n):
        rows = []
        for i, (name, addr, country) in enumerate(entities[:n]):
            rows.append({'entity_id': f'{id_prefix}{i+1:05d}',
                         'business_name': name, 'business_address': addr, 'country': country})
        return pd.DataFrame(rows)

    def make_source23(entities, id_prefix, n_entities, n_records, singleton_indices):
        rows, mapping = [], {}
        rid = 1
        for i, (name, addr, country) in enumerate(entities[:n_entities]):
            if i in singleton_indices:
                continue  # S1 singletons have no matches
            n_matches = random.randint(1, 3)
            matched_ids = []
            for _ in range(n_matches):
                eid = f'{id_prefix}{rid:05d}'
                rows.append({'entity_id': eid,
                             'business_name': perturb_name(name),
                             'business_address': perturb_address(addr),
                             'country': country})
                matched_ids.append(eid)
                rid += 1
            # add a hard negative (different entity, similar sounding)
            rows.append({'entity_id': f'{id_prefix}{rid:05d}',
                         'business_name': perturb_name(random.choice(entities)[0]),
                         'business_address': perturb_address(random.choice(entities)[1]),
                         'country': country})
            rid += 1
            s1_id = f'S1-{i+1:05d}'
            mapping[s1_id] = matched_ids
        return pd.DataFrame(rows), mapping

    SINGLETON_IDX = {10, 11}  # last 2 entities are singletons
    N = len(TRUE_ENTITIES)

    # -- TRAIN --------------------------------------------------
    s1_train = make_source1(TRUE_ENTITIES, 'S1-', N)
    s2_train, gt2 = make_source23(TRUE_ENTITIES, 'S2-', N, 3, SINGLETON_IDX)
    s3_train, gt3 = make_source23(TRUE_ENTITIES, 'S3-', N, 2, SINGLETON_IDX)
    # Merge ground truth
    gt_rows = []
    for s1_id in s1_train['entity_id']:
        m2 = gt2.get(s1_id, [])
        m3 = gt3.get(s1_id, [])
        gt_rows.append({'source1_entity_id': s1_id,
                         'matched_entity_ids': ','.join(m2 + m3)})
    gt_train = pd.DataFrame(gt_rows)

    # -- TEST (use last ~30% of entities as test) ---------------
    TEST_ENTITIES = [
        ('Pinnacle Tech Group Inc',     '777 Innovation Dr, Boston, MA 02101', 'US'),
        ('Rajasthan Handicrafts Pvt',   'Johari Bazaar, Jaipur 302001',        'India'),
        ('Bordeaux Vignobles SAS',      '15 Route du Médoc, 33000 Bordeaux',   'France'),
        ('CloudNine Logistics LLC',     '88 Harbor View Blvd, Seattle WA 98101', 'US'),
        ('Standalone Business Co',      '1 Isolated Road, Nowhere, MT 59001',  'US'),  # singleton
    ]
    SINGLETON_TEST = {4}
    s1_test = make_source1(TEST_ENTITIES, 'S1-', len(TEST_ENTITIES))
    s2_test, _ = make_source23(TEST_ENTITIES, 'S2-', len(TEST_ENTITIES), 2, SINGLETON_TEST)
    s3_test, _ = make_source23(TEST_ENTITIES, 'S3-', len(TEST_ENTITIES), 2, SINGLETON_TEST)

    # -- Write to disk ------------------------------------------
    def write_tsv(df, path):
        df.to_csv(path, sep='\t', index=False, encoding='utf-8')

    write_tsv(s1_train, os.path.join(TRAIN_DIR, 'train_source1.tsv'))
    write_tsv(s2_train, os.path.join(TRAIN_DIR, 'train_source2.tsv'))
    write_tsv(s3_train, os.path.join(TRAIN_DIR, 'train_source3.tsv'))
    write_tsv(gt_train, os.path.join(TRAIN_DIR, 'train_ground_truth.tsv'))
    write_tsv(s1_test,  os.path.join(TEST_DIR,  'test_source1.tsv'))
    write_tsv(s2_test,  os.path.join(TEST_DIR,  'test_source2.tsv'))
    write_tsv(s3_test,  os.path.join(TEST_DIR,  'test_source3.tsv'))

    print(f'[OK] Dummy dataset written:')
    print(f'   Train -- S1: {len(s1_train)}, S2: {len(s2_train)}, S3: {len(s3_train)}')
    print(f'   Test  -- S1: {len(s1_test)},  S2: {len(s2_test)},  S3: {len(s3_test)}')
    return s1_train, s2_train, s3_train, gt_train, s1_test, s2_test, s3_test


if USE_DUMMY_DATA:
    create_dummy_dataset()

## 📂 5. Data Loading & Preprocessing

In [ ]:
def load_tsv(path: str) -> pd.DataFrame:
    return pd.read_csv(path, sep='\t', dtype=str, keep_default_na=False)


def load_all_data():
    print('Loading data...')
    tr_s1 = preprocess_df(load_tsv(os.path.join(TRAIN_DIR, 'train_source1.tsv')))
    tr_s2 = preprocess_df(load_tsv(os.path.join(TRAIN_DIR, 'train_source2.tsv')))
    tr_s3 = preprocess_df(load_tsv(os.path.join(TRAIN_DIR, 'train_source3.tsv')))
    gt    = load_tsv(os.path.join(TRAIN_DIR, 'train_ground_truth.tsv'))
    te_s1 = preprocess_df(load_tsv(os.path.join(TEST_DIR,  'test_source1.tsv')))
    te_s2 = preprocess_df(load_tsv(os.path.join(TEST_DIR,  'test_source2.tsv')))
    te_s3 = preprocess_df(load_tsv(os.path.join(TEST_DIR,  'test_source3.tsv')))

    print(f'  Train -- S1: {len(tr_s1)}, S2: {len(tr_s2)}, S3: {len(tr_s3)}')
    print(f'  Test  -- S1: {len(te_s1)}, S2: {len(te_s2)}, S3: {len(te_s3)}')
    return tr_s1, tr_s2, tr_s3, gt, te_s1, te_s2, te_s3


tr_s1, tr_s2, tr_s3, gt, te_s1, te_s2, te_s3 = load_all_data()

# Build quick lookup dicts: entity_id -> row
def build_index(df: pd.DataFrame) -> Dict[str, dict]:
    return df.set_index('entity_id').to_dict('index')

tr_s1_idx = build_index(tr_s1)
tr_s2_idx = build_index(tr_s2)
tr_s3_idx = build_index(tr_s3)
te_s1_idx = build_index(te_s1)
te_s23_idx = {**build_index(te_s2), **build_index(te_s3)}

# Parse ground truth -> dict {s1_id: set(matched_ids)}
def parse_gt(df: pd.DataFrame) -> Dict[str, Set[str]]:
    result = {}
    for _, row in df.iterrows():
        s1_id  = row['source1_entity_id']
        matched = str(row.get('matched_entity_ids', '')).strip()
        result[s1_id] = set(matched.split(',')) - {''} if matched else set()
    return result

gt_dict = parse_gt(gt)
print(f'  Ground truth: {len(gt_dict)} S1 entities, '
      f'{sum(len(v) for v in gt_dict.values())} total matches')

## 📐 6. Evaluation Metric: F_0.5 (macro-averaged)

In [ ]:
def f_beta(precision: float, recall: float, beta: float = 0.5) -> float:
    if precision + recall == 0:
        return 0.0
    b2 = beta ** 2
    return (1 + b2) * precision * recall / (b2 * precision + recall)


def score_predictions(
    pred_dict: Dict[str, Set[str]],
    gold_dict: Dict[str, Set[str]],
    beta: float = 0.5,
) -> dict:
    """
    Macro-averaged F_β score.
    pred_dict : {s1_id -> set of predicted matched ids}
    gold_dict : {s1_id -> set of true matched ids}
    """
    scores, p_list, r_list = [], [], []
    for s1_id, gold in gold_dict.items():
        pred = pred_dict.get(s1_id, set())
        if len(gold) == 0 and len(pred) == 0:
            # Singleton correctly predicted
            scores.append(1.0); p_list.append(1.0); r_list.append(1.0)
        elif len(gold) == 0:
            # Singleton falsely given matches
            scores.append(0.0); p_list.append(0.0); r_list.append(1.0)
        elif len(pred) == 0:
            # Has matches but predicted none
            scores.append(0.0); p_list.append(1.0); r_list.append(0.0)
        else:
            tp = len(gold & pred)
            p  = tp / len(pred)
            r  = tp / len(gold)
            f  = f_beta(p, r, beta)
            scores.append(f); p_list.append(p); r_list.append(r)
    return {
        'f_05':      np.mean(scores),
        'precision': np.mean(p_list),
        'recall':    np.mean(r_list),
        'n':         len(scores),
    }


print('[OK] Evaluation metric defined')

## 🧮 7. Feature Engineering

Shared between Phase 1 & Phase 2 — computes pairwise similarity features for a (S1, S2/S3) candidate pair.

In [ ]:
def jaccard_tokens(a: str, b: str) -> float:
    sa, sb = set(a.split()), set(b.split())
    if not sa and not sb: return 1.0
    if not sa or not sb:  return 0.0
    return len(sa & sb) / len(sa | sb)


def token_sort_ratio(a: str, b: str) -> float:
    return fuzz.token_sort_ratio(a, b) / 100.0


def token_set_ratio(a: str, b: str) -> float:
    return fuzz.token_set_ratio(a, b) / 100.0


def char_ngram_jaccard(a: str, b: str, n: int = 3) -> float:
    if len(a) < n and len(b) < n: return 1.0
    sa = set(a[i:i+n] for i in range(len(a)-n+1))
    sb = set(b[i:i+n] for i in range(len(b)-n+1))
    if not sa and not sb: return 1.0
    if not sa or not sb:  return 0.0
    return len(sa & sb) / len(sa | sb)


def len_ratio(a: str, b: str) -> float:
    la, lb = len(a), len(b)
    if la == 0 and lb == 0: return 1.0
    return min(la, lb) / max(la, lb) if max(la, lb) > 0 else 0.0


def common_token_count(a: str, b: str) -> int:
    return len(set(a.split()) & set(b.split()))


def extract_numbers(text: str) -> Set[str]:
    """Extract numeric tokens (useful for house numbers, zip codes)."""
    return set(re.findall(r'\b\d+\b', text))


def number_overlap(a: str, b: str) -> float:
    na, nb = extract_numbers(a), extract_numbers(b)
    if not na and not nb: return 1.0
    if not na or  not nb: return 0.0
    return len(na & nb) / len(na | nb)


FEATURE_NAMES = [
    'name_jaccard', 'name_token_sort', 'name_token_set',
    'name_char3_jaccard', 'name_len_ratio', 'name_common_tokens',
    'addr_jaccard', 'addr_token_sort', 'addr_token_set',
    'addr_char3_jaccard', 'addr_len_ratio', 'addr_number_overlap',
    'comb_jaccard', 'comb_token_sort',
    'country_match',
    'name_empty_a', 'name_empty_b', 'addr_empty_a', 'addr_empty_b',
]


def compute_features(
    row_a: dict,   # S1 record (must have clean_name, clean_address, country)
    row_b: dict,   # S2/S3 record
) -> List[float]:
    na, nb = row_a.get('clean_name',''),   row_b.get('clean_name','')
    aa, ab = row_a.get('clean_address',''), row_b.get('clean_address','')
    ca, cb = row_a.get('country',''),       row_b.get('country','')
    combo_a = make_combined_text(na, aa)
    combo_b = make_combined_text(nb, ab)

    return [
        # Name features
        jaccard_tokens(na, nb),
        token_sort_ratio(na, nb),
        token_set_ratio(na, nb),
        char_ngram_jaccard(na, nb, 3),
        len_ratio(na, nb),
        float(common_token_count(na, nb)),
        # Address features
        jaccard_tokens(aa, ab),
        token_sort_ratio(aa, ab),
        token_set_ratio(aa, ab),
        char_ngram_jaccard(aa, ab, 3),
        len_ratio(aa, ab),
        number_overlap(aa, ab),
        # Combined text
        jaccard_tokens(combo_a, combo_b),
        token_sort_ratio(combo_a, combo_b),
        # Country match
        float(ca == cb and ca != ''),
        # Missingness indicators
        float(na == ''), float(nb == ''),
        float(aa == ''), float(ab == ''),
    ]


assert len(FEATURE_NAMES) == len(compute_features(
    {'clean_name': 'acme corp', 'clean_address': '123 main st', 'country': 'US'},
    {'clean_name': 'acme corporation', 'clean_address': '123 main street', 'country': 'US'}
))
print(f'[OK] Feature engineering ready -- {len(FEATURE_NAMES)} features')

---
# 🚀 PHASE 1 — Fast Baseline
## TF-IDF Blocking + LightGBM Classifier
### Expected F_0.5: 0.75-0.85 | Train time: ~30 min on real data

### P1.1 — TF-IDF Blocking

Generates candidate pairs efficiently using cosine similarity on TF-IDF vectors.

In [ ]:
class TFIDFBlocker:
    """
    Fast candidate generation using TF-IDF + sparse cosine similarity.
    Returns top-K candidate S2/S3 ids for each S1 entity.
    """

    def __init__(self, max_features=100_000, ngram_range=(1,2), top_k=50):
        self.vectorizer = TfidfVectorizer(
            max_features=max_features,
            ngram_range=ngram_range,
            analyzer='word',
            sublinear_tf=True,
            min_df=1,
        )
        self.top_k = top_k
        self.s23_ids   = None
        self.s23_matrix = None

    def fit_s23(self, s2: pd.DataFrame, s3: pd.DataFrame):
        """Fit vectorizer and index S2+S3 corpus."""
        s23 = pd.concat([s2, s3], ignore_index=True)
        self.s23_ids = list(s23['entity_id'])
        corpus = list(s23['combined'].fillna(''))
        print(f'  Fitting TF-IDF on {len(corpus):,} S2/S3 records...')
        t0 = time.time()
        self.vectorizer.fit(corpus)
        self.s23_matrix = self.vectorizer.transform(corpus)  # (N_s23, vocab)
        print(f'  Done in {time.time()-t0:.1f}s | Matrix: {self.s23_matrix.shape}')

    def get_candidates(
        self, s1: pd.DataFrame, batch_size: int = 1000
    ) -> Dict[str, List[Tuple[str, float]]]:
        """
        Returns {s1_id: [(s23_id, cos_sim), ...]} sorted by descending similarity.
        Uses batched matrix multiplication for efficiency.
        """
        results = {}
        s1_ids    = list(s1['entity_id'])
        s1_corpus = list(s1['combined'].fillna(''))
        n = len(s1_ids)
        print(f'  Blocking {n:,} S1 entities (batch={batch_size}, K={self.top_k})...')
        t0 = time.time()

        for start in tqdm(range(0, n, batch_size), desc='TF-IDF blocks'):
            batch_texts = s1_corpus[start:start+batch_size]
            batch_ids   = s1_ids[start:start+batch_size]
            q_mat = self.vectorizer.transform(batch_texts)  # (B, vocab)

            # cosine similarity (both matrices are L2-normalised by TF-IDF defaults)
            # We need to L2-normalise query and index
            from sklearn.preprocessing import normalize
            q_norm = normalize(q_mat)
            d_norm = normalize(self.s23_matrix)
            sims = (q_norm @ d_norm.T).toarray()  # (B, N_s23)

            # top-K per query
            k = min(self.top_k, sims.shape[1])
            top_idx = np.argpartition(sims, -k, axis=1)[:, -k:]
            for i, s1_id in enumerate(batch_ids):
                row = [(self.s23_ids[j], float(sims[i, j])) for j in top_idx[i]]
                row.sort(key=lambda x: -x[1])
                results[s1_id] = row

        print(f'  Blocking done in {time.time()-t0:.1f}s')
        return results

    def save(self, path: str):
        joblib.dump({'vectorizer': self.vectorizer,
                     's23_ids': self.s23_ids,
                     's23_matrix': self.s23_matrix,
                     'top_k': self.top_k}, path)

    @classmethod
    def load(cls, path: str) -> 'TFIDFBlocker':
        data = joblib.load(path)
        obj = cls(top_k=data['top_k'])
        obj.vectorizer  = data['vectorizer']
        obj.s23_ids     = data['s23_ids']
        obj.s23_matrix  = data['s23_matrix']
        return obj


print('[OK] TFIDFBlocker class defined')

### P1.2 — Training Data Construction

In [ ]:
def build_training_pairs(
    s1: pd.DataFrame,
    s23_idx: Dict[str, dict],
    gt: Dict[str, Set[str]],
    candidates: Dict[str, List[Tuple[str, float]]],
    s1_idx: Dict[str, dict],
    neg_ratio: float = 3.0,
) -> pd.DataFrame:
    """
    Build labelled pair features from blocking candidates.
    Positive: (S1, S23) where S23 is in gt[S1]
    Negative: (S1, S23) from candidates but not in gt[S1]
    neg_ratio: negatives per positive (to handle class imbalance)
    """
    import random
    random.seed(RANDOM_SEED)

    rows_X, rows_y, rows_meta = [], [], []
    n_pos = n_neg = n_miss = 0

    for _, s1_row in tqdm(s1.iterrows(), total=len(s1), desc='Building pairs'):
        s1_id  = s1_row['entity_id']
        gold   = gt.get(s1_id, set())
        cands  = candidates.get(s1_id, [])
        cand_ids = [c[0] for c in cands]

        # Positives: gold matches that appear in candidates
        found_in_cands = gold & set(cand_ids)
        # Inject all gold matches (even if not in candidates) to avoid recall ceiling issues
        for s23_id in gold:
            if s23_id not in s23_idx:
                n_miss += 1
                continue
            feats = compute_features(dict(s1_row), s23_idx[s23_id])
            rows_X.append(feats); rows_y.append(1)
            rows_meta.append({'s1_id': s1_id, 's23_id': s23_id})
            n_pos += 1

        # Negatives from candidates
        neg_cands = [c for c in cand_ids if c not in gold]
        max_neg = int(len(gold) * neg_ratio) if gold else max(1, int(neg_ratio))
        random.shuffle(neg_cands)
        for s23_id in neg_cands[:max_neg]:
            if s23_id not in s23_idx:
                continue
            feats = compute_features(dict(s1_row), s23_idx[s23_id])
            rows_X.append(feats); rows_y.append(0)
            rows_meta.append({'s1_id': s1_id, 's23_id': s23_id})
            n_neg += 1

    df = pd.DataFrame(rows_X, columns=FEATURE_NAMES)
    df['label']   = rows_y
    df['s1_id']   = [m['s1_id']  for m in rows_meta]
    df['s23_id']  = [m['s23_id'] for m in rows_meta]
    print(f'  Pairs -- positives: {n_pos}, negatives: {n_neg}, gold not in S23: {n_miss}')
    return df


print('[OK] Training pair builder defined')

### P1.3 — Train/Validation Split & Blocking

In [ ]:
# -- Train/Val split on S1 entity level (not pair level!) ------
from sklearn.model_selection import GroupShuffleSplit

s1_ids_all = list(tr_s1['entity_id'])
n_val = max(1, int(len(s1_ids_all) * VAL_FRACTION))
s1_train_ids = s1_ids_all[:-n_val]
s1_val_ids   = s1_ids_all[-n_val:]

tr_s1_train = tr_s1[tr_s1['entity_id'].isin(set(s1_train_ids))].reset_index(drop=True)
tr_s1_val   = tr_s1[tr_s1['entity_id'].isin(set(s1_val_ids))].reset_index(drop=True)

print(f'Train S1: {len(tr_s1_train)}, Val S1: {len(tr_s1_val)}')

# -- Fit TF-IDF blocker on training S2+S3 ---------------------
print('\n-- Phase 1: TF-IDF Blocking --')
tr_s23 = pd.concat([tr_s2, tr_s3], ignore_index=True)
tr_s23_idx = {**build_index(tr_s2), **build_index(tr_s3)}

p1_blocker = TFIDFBlocker(
    max_features=P1_TFIDF_MAX_FEATURES,
    ngram_range=P1_TFIDF_NGRAM_RANGE,
    top_k=P1_TOP_K_CANDIDATES,
)
p1_blocker.fit_s23(tr_s2, tr_s3)

# Get candidates for both train and val splits
print('  Getting train candidates...')
tr_candidates = p1_blocker.get_candidates(tr_s1_train)
print('  Getting val candidates...')
val_candidates = p1_blocker.get_candidates(tr_s1_val)

In [ ]:
# -- Check blocking recall --------------------------------------
def compute_blocking_recall(gt_dict, candidates, split_ids):
    hit = total = 0
    for s1_id in split_ids:
        gold = gt_dict.get(s1_id, set())
        if not gold: continue
        cand_set = set(c[0] for c in candidates.get(s1_id, []))
        hit   += len(gold & cand_set)
        total += len(gold)
    return hit / total if total else 0.0

tr_recall  = compute_blocking_recall(gt_dict, tr_candidates,  s1_train_ids)
val_recall = compute_blocking_recall(gt_dict, val_candidates, s1_val_ids)
print(f'  Blocking recall -- Train: {tr_recall:.3f} | Val: {val_recall:.3f}')
print(f'  (This is the upper bound on matcher recall)')

### P1.4 — Feature Extraction & LightGBM Training

In [ ]:
print('Building training pairs...')
train_pairs = build_training_pairs(
    tr_s1_train, tr_s23_idx, gt_dict, tr_candidates, tr_s1_idx
)
print(f'  Total training pairs: {len(train_pairs):,}  |  '
      f'Positive rate: {train_pairs["label"].mean():.3f}')

print('Building validation pairs...')
val_pairs = build_training_pairs(
    tr_s1_val, tr_s23_idx, gt_dict, val_candidates, tr_s1_idx
)
print(f'  Total validation pairs: {len(val_pairs):,}  |  '
      f'Positive rate: {val_pairs["label"].mean():.3f}')

In [ ]:
print('\n-- Training LightGBM --')

X_tr = train_pairs[FEATURE_NAMES].values
y_tr = train_pairs['label'].values
X_va = val_pairs[FEATURE_NAMES].values
y_va = val_pairs['label'].values

p1_lgbm = lgb.LGBMClassifier(**P1_LGBM_PARAMS)

p1_lgbm.fit(
    X_tr, y_tr,
    eval_set=[(X_va, y_va)],
    callbacks=[
        lgb.early_stopping(50, verbose=False),
        lgb.log_evaluation(100),
    ]
)

# Feature importance
imp = pd.Series(p1_lgbm.feature_importances_, index=FEATURE_NAMES).sort_values(ascending=False)
print('\nTop 10 features:')
print(imp.head(10).to_string())
joblib.dump(p1_lgbm, os.path.join(MODEL_DIR, 'p1_lgbm.pkl'))
print(f'\n[OK] LightGBM saved -> {MODEL_DIR}/p1_lgbm.pkl')

### P1.5 — Threshold Tuning on Validation Set

In [ ]:
def build_predictions(
    s1: pd.DataFrame,
    s23_idx: Dict[str, dict],
    candidates: Dict[str, List[Tuple[str, float]]],
    model,
    threshold: float,
    include_tfidf_score: bool = False,
    extra_feature: Optional[np.ndarray] = None,  # per-pair extra scores (Phase 2)
) -> Tuple[Dict[str, Set[str]], Dict[str, List[str]]]:
    """
    Run model on all candidate pairs and apply threshold.
    Returns:
        matched   : {s1_id -> set(matched_ids)}  (final predictions)
        all_cands : {s1_id -> [all candidate ids]}  (for candidate_pairs.tsv)
    """
    matched   = {}
    all_cands = {}
    s1_idx_local = {row['entity_id']: dict(row) for _, row in s1.iterrows()}

    for _, s1_row in tqdm(s1.iterrows(), total=len(s1), desc='Predicting'):
        s1_id  = s1_row['entity_id']
        cands  = candidates.get(s1_id, [])

        all_cands[s1_id] = [c[0] for c in cands]

        if not cands:
            matched[s1_id] = set()
            continue

        feats_list = []
        valid_cands = []
        for s23_id, _ in cands:
            if s23_id not in s23_idx: continue
            feats_list.append(compute_features(dict(s1_row), s23_idx[s23_id]))
            valid_cands.append(s23_id)

        if not feats_list:
            matched[s1_id] = set()
            continue

        X = np.array(feats_list)
        probs = model.predict_proba(X)[:, 1]
        matched[s1_id] = {
            s23_id
            for s23_id, prob in zip(valid_cands, probs)
            if prob >= threshold
        }

    return matched, all_cands


# -- Tune threshold on val set ---------------------------------
print('Predicting on validation set...')
val_matched, val_cands = build_predictions(
    tr_s1_val, tr_s23_idx, val_candidates, p1_lgbm, P1_MATCH_THRESHOLD
)
val_gt = {k: gt_dict[k] for k in s1_val_ids if k in gt_dict}
metrics = score_predictions(val_matched, val_gt)
print(f'\nValidation Metrics @ threshold={P1_MATCH_THRESHOLD:.2f}:')
print(f"  F_0.5:     {metrics['f_05']:.4f}")
print(f"  Precision: {metrics['precision']:.4f}")
print(f"  Recall:    {metrics['recall']:.4f}")

# Grid search threshold
print('\nSearching best threshold...')
best_thresh, best_f05 = P1_MATCH_THRESHOLD, metrics['f_05']
# Collect all probs on val set for efficient search
for thresh in np.arange(0.25, 0.75, 0.05):
    vm, _ = build_predictions(tr_s1_val, tr_s23_idx, val_candidates, p1_lgbm, float(thresh))
    m = score_predictions(vm, val_gt)
    print(f'  thresh={thresh:.2f} -> F_0.5={m["f_05"]:.4f}  P={m["precision"]:.3f}  R={m["recall"]:.3f}')
    if m['f_05'] > best_f05:
        best_f05, best_thresh = m['f_05'], float(thresh)

print(f'\n[OK] Best threshold: {best_thresh:.2f}  (F_0.5={best_f05:.4f})')
P1_MATCH_THRESHOLD = best_thresh

### P1.6 — Full Train (retrain on all training data) & Test Inference

In [ ]:
print('\n-- Retraining on full training data --')

# Rebuild blocker on full train S2/S3
p1_blocker_full = TFIDFBlocker(
    max_features=P1_TFIDF_MAX_FEATURES,
    ngram_range=P1_TFIDF_NGRAM_RANGE,
    top_k=P1_TOP_K_CANDIDATES,
)
p1_blocker_full.fit_s23(tr_s2, tr_s3)
tr_all_cands = p1_blocker_full.get_candidates(tr_s1)

full_pairs = build_training_pairs(
    tr_s1, tr_s23_idx, gt_dict, tr_all_cands, tr_s1_idx
)
X_full = full_pairs[FEATURE_NAMES].values
y_full = full_pairs['label'].values

p1_lgbm_full = lgb.LGBMClassifier(**{**P1_LGBM_PARAMS,
                                      'n_estimators': p1_lgbm.best_iteration_ or P1_LGBM_PARAMS['n_estimators']})
p1_lgbm_full.fit(X_full, y_full, callbacks=[lgb.log_evaluation(100)])
joblib.dump(p1_lgbm_full, os.path.join(MODEL_DIR, 'p1_lgbm_full.pkl'))
print('[OK] Full model trained & saved')

In [ ]:
print('\n-- Phase 1: Test Set Inference --')

# Fit a combined TF-IDF on train+test S2/S3 for test blocking
# (TF-IDF is unsupervised so this is fine / not data leakage)
all_s23_for_test = pd.concat([tr_s2, tr_s3, te_s2, te_s3], ignore_index=True)

p1_blocker_test = TFIDFBlocker(
    max_features=P1_TFIDF_MAX_FEATURES,
    ngram_range=P1_TFIDF_NGRAM_RANGE,
    top_k=P1_TOP_K_CANDIDATES,
)
# Only index test S2/S3 for test blocking
p1_blocker_test.fit_s23(te_s2, te_s3)
te_candidates = p1_blocker_test.get_candidates(te_s1)
p1_blocker_test.save(os.path.join(MODEL_DIR, 'p1_blocker_test.pkl'))

te_matched_p1, te_cands_p1 = build_predictions(
    te_s1, te_s23_idx, te_candidates, p1_lgbm_full, P1_MATCH_THRESHOLD
)

# Stats
n_matched = sum(len(v) for v in te_matched_p1.values())
n_singletons = sum(1 for v in te_matched_p1.values() if not v)
print(f'  Test predictions: {n_matched} total matches, {n_singletons} singletons')

### P1.7 — Write Output Files

In [ ]:
def write_matching_results(
    matched: Dict[str, Set[str]],
    s1_ids: List[str],
    path: str,
):
    """Write matching_results.tsv -- one row per S1 entity."""
    rows = []
    for s1_id in s1_ids:
        ids = sorted(matched.get(s1_id, set()))  # sorted for reproducibility
        rows.append({'source1_entity_id': s1_id,
                     'matched_entity_ids': ','.join(ids)})
    df = pd.DataFrame(rows)
    df.to_csv(path, sep='\t', index=False, encoding='utf-8')
    print(f'  Wrote {len(df)} rows -> {path}')


def write_candidate_pairs(
    all_cands: Dict[str, List[str]],
    s1_ids: List[str],
    path: str,
):
    """Write candidate_pairs.tsv -- one row per S1 entity."""
    rows = []
    for s1_id in s1_ids:
        ids = list(dict.fromkeys(all_cands.get(s1_id, [])))  # dedup, preserve order
        rows.append({'source1_entity_id': s1_id,
                     'candidate_entity_ids': ','.join(ids)})
    df = pd.DataFrame(rows)
    df.to_csv(path, sep='\t', index=False, encoding='utf-8')
    print(f'  Wrote {len(df)} rows -> {path}')


TE_S1_IDS = list(te_s1['entity_id'])

P1_MATCHING_PATH  = os.path.join(OUTPUT_DIR, 'matching_results.tsv')
P1_CANDIDATE_PATH = os.path.join(OUTPUT_DIR, 'candidate_pairs.tsv')

print('\n-- Writing Phase 1 output files --')
write_matching_results(te_matched_p1, TE_S1_IDS, P1_MATCHING_PATH)
write_candidate_pairs(te_cands_p1, TE_S1_IDS, P1_CANDIDATE_PATH)
print('[OK] Phase 1 output files written')

### P1.8 — Validate Submission (Official Validator)

In [ ]:
import subprocess, sys

def run_validator(matching_path: str, candidate_path: str, test_dir: str):
    """Run the official validate_submission.py and display output."""
    result = subprocess.run(
        [sys.executable, 'validate_submission.py',
         '--matching',   matching_path,
         '--candidate',  candidate_path,
         '--test-dir',   test_dir],
        capture_output=True, text=True
    )
    print('=' * 60)
    print('VALIDATOR OUTPUT:')
    print(result.stdout)
    if result.stderr:
        print('STDERR:', result.stderr)
    print('=' * 60)
    if result.returncode == 0:
        print('[OK] VALIDATION PASSED -- safe to submit!')
    else:
        print('[FAIL] VALIDATION FAILED -- fix issues above before submitting.')
    return result.returncode == 0


print('\n-- Running official validator on Phase 1 output --')
p1_valid = run_validator(P1_MATCHING_PATH, P1_CANDIDATE_PATH, TEST_DIR)

---
# 🔬 PHASE 2 — High-Performance Pipeline
## Multilingual Sentence Transformers + FAISS + Cross-Encoder Ensemble
### Expected F_0.5: 0.87-0.93 | Train time: 4-12 hours (GPU recommended)

> **Note:** Phase 2 builds on top of Phase 1. Run Phase 1 first. Skip this section if `PHASE == 1`.

In [ ]:
if PHASE < 2:
    print('PHASE=1 -- skipping Phase 2. Set PHASE=2 to run the full pipeline.')
    # Notebook will still define all Phase 2 cells but skip execution
    _skip_p2 = True
else:
    _skip_p2 = False
    print('PHASE=2 -- running full pipeline')

### P2.1 — Multilingual Sentence Transformer Encoder

In [ ]:
# Note: The %%skip_if magic is defined in a helper below;
# alternatively just wrap in: if not _skip_p2:

if not _skip_p2:
    from sentence_transformers import SentenceTransformer
    import faiss

    print(f'Loading sentence transformer: {P2_EMBED_MODEL}')
    print(f'  Model: paraphrase-multilingual-mpnet-base-v2  (278M params, Apache 2.0)')
    embed_model = SentenceTransformer(P2_EMBED_MODEL, device=DEVICE)
    embed_dim   = embed_model.get_sentence_embedding_dimension()
    print(f'  Embedding dim: {embed_dim}  |  Device: {DEVICE}')


    def encode_df(df: pd.DataFrame, model: SentenceTransformer,
                  batch_size: int = 128, show_progress: bool = True) -> np.ndarray:
        """
        Encode the 'combined' field of a dataframe.
        Returns L2-normalised float32 embeddings.
        """
        texts = list(df['combined'].fillna(''))
        embs  = model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=show_progress,
            normalize_embeddings=True,  # L2 norm for cosine sim via inner product
            convert_to_numpy=True,
        )
        return embs.astype(np.float32)


    print('[OK] Sentence transformer loaded')

### P2.2 — FAISS Indexing & Dense Candidate Retrieval

In [ ]:
if not _skip_p2:
    class FAISSBlocker:
        """
        ANN candidate generation using FAISS IndexFlatIP (exact inner product = cosine
        on L2-normalised embeddings). Falls back to IndexIVFFlat for large datasets.
        """

        def __init__(self, dim: int, top_k: int = 100, use_ivf_threshold: int = 50_000):
            self.dim = dim
            self.top_k = top_k
            self.use_ivf_threshold = use_ivf_threshold
            self.index = None
            self.s23_ids = None

        def build(self, s23: pd.DataFrame, embeddings: np.ndarray):
            n = len(s23)
            self.s23_ids = list(s23['entity_id'])
            if n > self.use_ivf_threshold:
                nlist = min(4096, n // 10)
                quantizer = faiss.IndexFlatIP(self.dim)
                self.index = faiss.IndexIVFFlat(quantizer, self.dim, nlist, faiss.METRIC_INNER_PRODUCT)
                self.index.train(embeddings)
                self.index.nprobe = min(256, nlist)
                print(f'  Using IVFFlat (nlist={nlist}, nprobe={self.index.nprobe})')
            else:
                self.index = faiss.IndexFlatIP(self.dim)
                print(f'  Using FlatIP (exact search)')
            self.index.add(embeddings)
            print(f'  FAISS index built: {self.index.ntotal:,} vectors')

        def search(
            self, query_embs: np.ndarray, s1_ids: List[str]
        ) -> Dict[str, List[Tuple[str, float]]]:
            k = min(self.top_k, self.index.ntotal)
            D, I = self.index.search(query_embs, k)  # (N_q, k)
            results = {}
            for i, s1_id in enumerate(s1_ids):
                cands = []
                for j in range(k):
                    idx = I[i, j]
                    if idx < 0: continue
                    cands.append((self.s23_ids[idx], float(D[i, j])))
                results[s1_id] = cands
            return results


    # Encode training S2/S3
    print('Encoding S2/S3 (train)...')
    tr_s23_df = pd.concat([tr_s2, tr_s3], ignore_index=True)
    tr_s23_embs = encode_df(tr_s23_df, embed_model, batch_size=P2_BATCH_SIZE)

    print('Encoding S1 (train)...')
    tr_s1_embs = encode_df(tr_s1, embed_model, batch_size=P2_BATCH_SIZE)

    print('Building FAISS index (train)...')
    p2_blocker_tr = FAISSBlocker(dim=embed_dim, top_k=P2_TOP_K_FAISS)
    p2_blocker_tr.build(tr_s23_df, tr_s23_embs)

    print('Retrieving P2 train candidates...')
    p2_tr_candidates = p2_blocker_tr.search(tr_s1_embs, list(tr_s1['entity_id']))

    p2_tr_recall = compute_blocking_recall(gt_dict, p2_tr_candidates, list(tr_s1['entity_id']))
    print(f'  FAISS blocking recall (train): {p2_tr_recall:.3f}')

### P2.3 — Merge FAISS + TF-IDF Candidates (Ensemble Blocking)

In [ ]:
if not _skip_p2:
    def merge_candidates(
        cands_a: Dict[str, List[Tuple[str, float]]],
        cands_b: Dict[str, List[Tuple[str, float]]],
        top_k: int = 150,
    ) -> Dict[str, List[Tuple[str, float]]]:
        """
        Union of two candidate dicts. Scores are kept from whichever source
        provided the candidate (or max if both did).
        """
        all_s1 = set(cands_a) | set(cands_b)
        merged = {}
        for s1_id in all_s1:
            score_map = {}
            for s23_id, score in cands_a.get(s1_id, []):
                score_map[s23_id] = max(score_map.get(s23_id, -1), score)
            for s23_id, score in cands_b.get(s1_id, []):
                score_map[s23_id] = max(score_map.get(s23_id, -1), score)
            merged[s1_id] = sorted(score_map.items(), key=lambda x: -x[1])[:top_k]
        return merged


    p2_tr_merged = merge_candidates(tr_all_cands, p2_tr_candidates, top_k=150)
    p2_tr_merged_recall = compute_blocking_recall(gt_dict, p2_tr_merged, list(tr_s1['entity_id']))
    print(f'Merged blocking recall: {p2_tr_merged_recall:.3f}')
    print(f'  TF-IDF recall:  {tr_recall:.3f}')
    print(f'  FAISS recall:   {p2_tr_recall:.3f}')

### P2.4 — Cross-Encoder Reranking

In [ ]:
if not _skip_p2:
    from sentence_transformers.cross_encoder import CrossEncoder

    print(f'Loading cross-encoder: {P2_CROSS_MODEL}')
    cross_encoder = CrossEncoder(P2_CROSS_MODEL, device=DEVICE, max_length=256)
    print('[OK] Cross-encoder loaded')


    def get_cross_encoder_scores(
        s1: pd.DataFrame,
        s23_idx: Dict[str, dict],
        candidates: Dict[str, List[Tuple[str, float]]],
        model: CrossEncoder,
        batch_size: int = 64,
    ) -> Dict[str, Dict[str, float]]:
        """
        Returns {s1_id: {s23_id: cross_encoder_score}} for all candidate pairs.
        """
        pairs_flat  = []   # list of (text_a, text_b)
        pair_meta   = []   # list of (s1_id, s23_id)
        s1_lookup   = {row['entity_id']: dict(row) for _, row in s1.iterrows()}

        for s1_id, cands in candidates.items():
            row_a = s1_lookup.get(s1_id)
            if row_a is None: continue
            text_a = row_a.get('combined', '')
            for s23_id, _ in cands:
                row_b = s23_idx.get(s23_id)
                if row_b is None: continue
                text_b = row_b.get('combined', '')
                pairs_flat.append((text_a, text_b))
                pair_meta.append((s1_id, s23_id))

        print(f'  Scoring {len(pairs_flat):,} pairs with cross-encoder...')
        scores_flat = cross_encoder.predict(
            pairs_flat,
            batch_size=batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
        )

        result = defaultdict(dict)
        for (s1_id, s23_id), score in zip(pair_meta, scores_flat):
            result[s1_id][s23_id] = float(score)
        return dict(result)


    print('Running cross-encoder on train candidates...')
    p2_ce_scores_tr = get_cross_encoder_scores(
        tr_s1, tr_s23_idx, p2_tr_merged, cross_encoder, batch_size=P2_BATCH_SIZE
    )
    print('[OK] Cross-encoder scoring done')

### P2.5 — Phase 2 Extended Feature Set & LightGBM

In [ ]:
if not _skip_p2:
    P2_FEATURE_NAMES = FEATURE_NAMES + [
        'faiss_score',        # embedding cosine similarity
        'tfidf_score',        # TF-IDF cosine similarity
        'cross_encoder_score', # cross-encoder relevance score
    ]

    def build_p2_training_pairs(
        s1: pd.DataFrame,
        s23_idx: Dict[str, dict],
        gt: Dict[str, Set[str]],
        merged_cands: Dict[str, List[Tuple[str, float]]],
        tfidf_cands: Dict[str, List[Tuple[str, float]]],
        faiss_cands: Dict[str, List[Tuple[str, float]]],
        ce_scores: Dict[str, Dict[str, float]],
        neg_ratio: float = 3.0,
    ) -> pd.DataFrame:
        import random; random.seed(RANDOM_SEED)
        rows_X, rows_y, rows_meta = [], [], []

        # Build quick score lookups
        tfidf_map = {s1_id: dict(cands) for s1_id, cands in tfidf_cands.items()}
        faiss_map = {s1_id: dict(cands) for s1_id, cands in faiss_cands.items()}

        for _, s1_row in tqdm(s1.iterrows(), total=len(s1), desc='P2 pairs'):
            s1_id = s1_row['entity_id']
            gold  = gt.get(s1_id, set())
            cands = merged_cands.get(s1_id, [])
            cand_ids = [c[0] for c in cands]

            all_ids = set(cand_ids) | gold
            positives = gold
            negatives = set(cand_ids) - gold
            max_neg = int(len(positives) * neg_ratio) if positives else max(1, int(neg_ratio))
            neg_sample = random.sample(list(negatives), min(max_neg, len(negatives)))

            for s23_id, label in [(i, 1) for i in positives] + [(i, 0) for i in neg_sample]:
                if s23_id not in s23_idx: continue
                base_feats = compute_features(dict(s1_row), s23_idx[s23_id])
                f_score = faiss_map.get(s1_id, {}).get(s23_id, 0.0)
                t_score = tfidf_map.get(s1_id, {}).get(s23_id, 0.0)
                c_score = ce_scores.get(s1_id, {}).get(s23_id, 0.0)
                rows_X.append(base_feats + [f_score, t_score, c_score])
                rows_y.append(label)
                rows_meta.append({'s1_id': s1_id, 's23_id': s23_id})

        df = pd.DataFrame(rows_X, columns=P2_FEATURE_NAMES)
        df['label']  = rows_y
        df['s1_id']  = [m['s1_id']  for m in rows_meta]
        df['s23_id'] = [m['s23_id'] for m in rows_meta]
        print(f'  P2 pairs -- positives: {sum(rows_y)}, negatives: {len(rows_y)-sum(rows_y)}')
        return df


    print('Building Phase 2 training pairs...')
    p2_train_pairs = build_p2_training_pairs(
        tr_s1, tr_s23_idx, gt_dict,
        p2_tr_merged, tr_all_cands, p2_tr_candidates, p2_ce_scores_tr
    )

    X2 = p2_train_pairs[P2_FEATURE_NAMES].values
    y2 = p2_train_pairs['label'].values

    p2_lgbm = lgb.LGBMClassifier(**P1_LGBM_PARAMS)
    # Train with cross-validation for better threshold estimation
    p2_lgbm.fit(X2, y2, callbacks=[lgb.log_evaluation(100)])
    joblib.dump(p2_lgbm, os.path.join(MODEL_DIR, 'p2_lgbm.pkl'))
    print('[OK] Phase 2 LightGBM trained & saved')

### P2.6 — Test Set Inference (Phase 2)

In [ ]:
if not _skip_p2:
    print('\n-- Phase 2: Test Set Inference --')

    # Encode test data
    print('Encoding test S2/S3...')
    te_s23_df  = pd.concat([te_s2, te_s3], ignore_index=True)
    te_s23_embs = encode_df(te_s23_df, embed_model, batch_size=P2_BATCH_SIZE)

    print('Encoding test S1...')
    te_s1_embs = encode_df(te_s1, embed_model, batch_size=P2_BATCH_SIZE)

    print('Building FAISS index (test)...')
    p2_blocker_te = FAISSBlocker(dim=embed_dim, top_k=P2_TOP_K_FAISS)
    p2_blocker_te.build(te_s23_df, te_s23_embs)
    p2_te_candidates = p2_blocker_te.search(te_s1_embs, TE_S1_IDS)

    print('Merging FAISS + TF-IDF test candidates...')
    p2_te_merged = merge_candidates(te_candidates, p2_te_candidates, top_k=150)

    print('Cross-encoder scoring on test candidates...')
    p2_ce_scores_te = get_cross_encoder_scores(
        te_s1, te_s23_idx, p2_te_merged, cross_encoder, batch_size=P2_BATCH_SIZE
    )

    # Build features and predict
    tfidf_te_map = {s1_id: dict(cands) for s1_id, cands in te_candidates.items()}
    faiss_te_map = {s1_id: dict(cands) for s1_id, cands in p2_te_candidates.items()}

    te_matched_p2 = {}
    te_all_cands_p2 = {}

    for _, s1_row in tqdm(te_s1.iterrows(), total=len(te_s1), desc='P2 test inference'):
        s1_id = s1_row['entity_id']
        cands = p2_te_merged.get(s1_id, [])
        te_all_cands_p2[s1_id] = [c[0] for c in cands]

        if not cands:
            te_matched_p2[s1_id] = set()
            continue

        feats_list, valid_ids = [], []
        for s23_id, _ in cands:
            if s23_id not in te_s23_idx: continue
            base = compute_features(dict(s1_row), te_s23_idx[s23_id])
            f_sc = faiss_te_map.get(s1_id, {}).get(s23_id, 0.0)
            t_sc = tfidf_te_map.get(s1_id, {}).get(s23_id, 0.0)
            c_sc = p2_ce_scores_te.get(s1_id, {}).get(s23_id, 0.0)
            feats_list.append(base + [f_sc, t_sc, c_sc])
            valid_ids.append(s23_id)

        if not feats_list:
            te_matched_p2[s1_id] = set()
            continue

        X = np.array(feats_list)
        probs = p2_lgbm.predict_proba(X)[:, 1]
        te_matched_p2[s1_id] = {s23_id for s23_id, p in zip(valid_ids, probs)
                                  if p >= P2_MATCH_THRESHOLD}

    n_m = sum(len(v) for v in te_matched_p2.values())
    n_s = sum(1 for v in te_matched_p2.values() if not v)
    print(f'P2 predictions: {n_m} matches, {n_s} singletons')

### P2.7 — Write & Validate Phase 2 Output

In [ ]:
if not _skip_p2:
    P2_MATCHING_PATH  = os.path.join(OUTPUT_DIR, 'matching_results.tsv')
    P2_CANDIDATE_PATH = os.path.join(OUTPUT_DIR, 'candidate_pairs.tsv')

    print('\n-- Writing Phase 2 output files --')
    write_matching_results(te_matched_p2, TE_S1_IDS, P2_MATCHING_PATH)
    write_candidate_pairs(te_all_cands_p2, TE_S1_IDS, P2_CANDIDATE_PATH)

    print('\n-- Running official validator on Phase 2 output --')
    p2_valid = run_validator(P2_MATCHING_PATH, P2_CANDIDATE_PATH, TEST_DIR)
else:
    print('Phase 2 skipped.')

---
## 📊 Final Summary

In [ ]:
print('=' * 60)
print('  SOLUTION SUMMARY')
print('=' * 60)
print(f'  Phase run:          {PHASE}')
print(f'  Dummy data used:    {USE_DUMMY_DATA}')
print(f'  Device:             {DEVICE}')
print()
print(f'  Validation F_0.5 (Phase 1): {metrics["f_05"]:.4f}')
print(f'  P1 threshold:       {P1_MATCH_THRESHOLD:.2f}')
print()
print(f'  Output files:')
print(f'    matching_results.tsv  -> {P1_MATCHING_PATH}')
print(f'    candidate_pairs.tsv   -> {P1_CANDIDATE_PATH}')
print()
print(f'  Models saved to: {MODEL_DIR}/')
print('=' * 60)
print('  [OK] Ready to submit matching_results.tsv to the leaderboard!')
print('  (Upload output/matching_results.tsv to the competition portal)')
print('=' * 60)